# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from tqdm.notebook import tqdm
import joblib
import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r"The `probability` parameter was deprecated.*"
)


## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [2]:
class FeatureExtractor(TransformerMixin, BaseEstimator):
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        hour = df["timestamp"].dt.hour
        weekday = df["timestamp"].dt.day_of_week
        df = df.drop(columns=["timestamp"])
        df["hour"] = hour
        df["dayofweek"] = weekday
        return df

In [3]:
class MyOneHotEncoder(TransformerMixin, BaseEstimator):
    def __init__(self, target):
        self.target = target

    def fit(self, X, y=None):
        X_dr = X.drop(columns=[self.target])

        self.cat_columns = X_dr.select_dtypes(
            include=["object", "category", "string", "bool"]
        ).columns

        self.encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

        if len(self.cat_columns) > 0:
            self.encoder.fit(X_dr[self.cat_columns])

        return self

    def transform(self, X):
        df = X.copy()
        y = df[self.target]
        X_dr = df.drop(columns=[self.target])

        if len(self.cat_columns) > 0:
            encoded = self.encoder.transform(X_dr[self.cat_columns])

            encoded_df = pd.DataFrame(
                encoded,
                columns=self.encoder.get_feature_names_out(),
                index=X_dr.index
            )

            X_dr = X_dr.drop(columns=self.cat_columns)
            X_dr = pd.concat([X_dr, encoded_df], axis=1)

        return X_dr, y

In [4]:
class TrainValidationTest:
    def split(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=21,
            stratify=y
        )

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train,
            y_train,
            test_size=0.2,
            random_state=21,
            stratify=y_train
        )

        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [5]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []
        self.best_model = None
        self.best_name = None

    def choose(self, X_train, y_train, X_valid, y_valid):
        self.results = []
        self.best_model = None
        self.best_name = None
        best_valid_score = -1

        for index, grid in enumerate(self.grids):
            name = self.grid_dict[index]
            print(f"Estimator: {name}")

            best_cv_score = -1
            best_params = None

            for params in tqdm(ParameterGrid(grid.param_grid)):
                model = clone(grid.estimator)
                model.set_params(**params)

                scores = cross_val_score(
                    model,
                    X_train,
                    y_train,
                    scoring=grid.scoring,
                    cv=grid.cv,
                    n_jobs=grid.n_jobs
                )
                score = scores.mean()

                if score > best_cv_score:
                    best_cv_score = score
                    best_params = params

            model = clone(grid.estimator)
            model.set_params(**best_params)
            model.fit(X_train, y_train)

            pred = model.predict(X_valid)
            valid_score = accuracy_score(y_valid, pred)

            self.results.append({
                "model": name,
                "params": best_params,
                "valid_score": valid_score
            })

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_cv_score:.5f}")
            print(f"Validation set accuracy score for best params: {valid_score:.5f}")

            if valid_score > best_valid_score:
                best_valid_score = valid_score
                self.best_model = model
                self.best_name = name

        print(f"Classifier with best validation set accuracy: {self.best_name}")
        return self.best_name

    def best_results(self):
        return pd.DataFrame(
            self.results,
            columns=["model", "params", "valid_score"]
        )


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [6]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        pred = self.estimator.predict(X_test)
        score = accuracy_score(y_test, pred)
        print(f"Accuracy of the final model is {score:.5f}")
        return score

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"Model was successfully saved to {path}")


## 4. Main program

1. Load the data from the file `checker_submits.csv`.
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [7]:
df = pd.read_csv("../data/checker_submits.csv")


In [8]:
preprocessing = Pipeline([
    ("feature_extractor", FeatureExtractor()),
    ("onehot_encoder", MyOneHotEncoder("dayofweek"))
])


In [9]:
X, y = preprocessing.fit_transform(df)


In [10]:
splitter = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X, y)


In [11]:
svm_params = {
    "kernel": ["linear", "rbf", "sigmoid"],
    "C": [0.01, 0.1, 1, 1.5, 5, 10],
    "gamma": ["scale", "auto"],
    "class_weight": ["balanced", None],
    "random_state": [21],
    "probability": [True]
}

tree_params = {
    "max_depth": range(1, 50),
    "class_weight": ["balanced", None],
    "criterion": ["entropy", "gini"],
    "random_state": [21]
}

rf_params = {
    "n_estimators": [5, 10, 50, 100],
    "max_depth": range(1, 50),
    "class_weight": ["balanced", None],
    "criterion": ["entropy", "gini"],
    "random_state": [21]
}


In [12]:
gs_svm = GridSearchCV(
    estimator=SVC(),
    param_grid=svm_params,
    scoring="accuracy",
    cv=2,
    n_jobs=1
)

gs_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(),
    param_grid=tree_params,
    scoring="accuracy",
    cv=2,
    n_jobs=1
)

gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=rf_params,
    scoring="accuracy",
    cv=2,
    n_jobs=1
)


In [13]:
selection = ModelSelection(
    grids=[gs_svm, gs_tree, gs_rf],
    grid_dict={
        0: "SVM",
        1: "Decision Tree",
        2: "Random Forest"
    }
)

best_name = selection.choose(X_train, y_train, X_valid, y_valid)


Estimator: SVM
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.77273
Validation set accuracy score for best params: 0.87778
Estimator: Decision Tree
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.80056
Validation set accuracy score for best params: 0.86296
Estimator: Random Forest
Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.85529
Validation set accuracy score for best params: 0.90370
Classifier with best validation set accuracy: Random Forest


  0%|          | 0/72 [00:00<?, ?it/s]

  0%|          | 0/196 [00:00<?, ?it/s]

  0%|          | 0/784 [00:00<?, ?it/s]

In [14]:
results = selection.best_results()
print(results.to_string(index=False, float_format="{:.5f}".format))


        model                                                                                                     params  valid_score
          SVM {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}      0.87778
Decision Tree                     {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}      0.86296
Random Forest       {'class_weight': None, 'criterion': 'gini', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}      0.90370


In [15]:
final = Finalize(selection.best_model)
score = final.final_score(X_train, y_train, X_test, y_test)


Accuracy of the final model is 0.91124


In [16]:
model_name = final.estimator.__class__.__name__
model_path = f"../data/{model_name}_{score:.5f}.sav"
final.save_model(model_path)


Model was successfully saved to ../data/RandomForestClassifier_0.91124.sav
